In [1]:
import cv2
from tensorflow.keras.models import load_model
import numpy as np

from performance_metrics.custom_metric import class_mAP, offset_MAE
from loss_function.custom_loss import AOILoss
from custom_layers.GridCenters import GridCenters

from input_encoder_decoder.output_decoder import decode_detections

%matplotlib inline

2023-10-23 16:35:34.345096: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2023-10-23 16:35:34.345143: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2023-10-23 16:35:34.345171: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2023-10-23 16:35:34.351889: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [3]:
model = load_model("saved_trained_models_v1/model_realistic_pcb_ssd6_conv6_256_15000_relu", custom_objects={'GridCenters': GridCenters,
                                                                                    'compute_loss': AOILoss,
                                                                                    'class_mAP': class_mAP,
                                                                                    'offset_MAE': offset_MAE})


2023-01-27 11:50:33.437830: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-01-27 11:50:33.441881: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-01-27 11:50:33.442021: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:981] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-01-27 11:50:33.442330: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorF

In [4]:
img_width = 256
img_height= 256

In [5]:
def plot_detections(decoded_pred, image):
    if decoded_pred.shape[0] == 0:
        return image
    
    #plt.figure(figsize=(10,6))
    #plt.imshow(image)
    #current_axis = plt.gca()

    #colors = plt.cm.hsv(np.linspace(0, 1, n_classes+1)).tolist() # Set the colors for the bounding boxes
    classes = ['background', 'ic'] # Just so we can print class names onto the image instead of IDs

    for label in decoded_pred:
        pred = np.array(label)
        corners = np.reshape(pred[2:], (-1, 2)).astype(int)
        #color = colors[int(pred[0])]
        label_text = '{}: {:.2f}'.format(classes[int(pred[0])], pred[1])
        for points in corners:
            image = cv2.circle(image, tuple(points), 2, (0,0,255), 1)
            #current_axis.add_patch(plt.Circle(tuple(points), 2, fill=False, color='red'))
        center_x = (corners[0, 0] + corners[3, 0]) / 2
        center_y = (corners[0, 1] + corners[3, 1]) / 2
        #current_axis.text(center_x, center_y, label, size='x-small', color='white', bbox={'facecolor':color, 'alpha':1.0})
        font = cv2.FONT_HERSHEY_SIMPLEX

        # fontScale
        fontScale = 1

        # Blue color in BGR
        color = (255, 0, 0)

        # Line thickness of 2 px
        thickness = 2
        
        image = cv2.putText(image, label_text, (int(center_x),int(center_y)), font, fontScale, color, thickness, cv2.LINE_AA)
        
    return image   

In [8]:
cam = cv2.VideoCapture(0)
while True:
    ret_val, img = cam.read()
    if ret_val:
        #print(img.shape) (480, 640, 3)
        img = img[0:479, 0:479]
        img = cv2.resize(img, (img_height,img_width), interpolation = cv2.INTER_AREA)
        img_pred = img.copy()
        img_pred = cv2.cvtColor(img_pred, cv2.COLOR_BGR2RGB)
        predictions = model.predict([np.expand_dims(img_pred, axis=0)], verbose=0)
        #print(predictions)
        decoded_pred = decode_detections(predictions, img_height=img_height, img_width=img_width)
        #print(decoded_pred[0])
        img = plot_detections(decoded_pred[0],img)
        #break
        img = cv2.resize(img, (img_height*2,img_width*2), interpolation = cv2.INTER_AREA)
        cv2.imshow('AI AOI', img)
    if cv2.waitKey(1) == 27: 
        break  # esc to quit
cv2.destroyAllWindows()

QObject::moveToThread: Current thread (0x1f9fea20) is not the object's thread (0x1fa15850).
Cannot move to target thread (0x1f9fea20)

QObject::moveToThread: Current thread (0x1f9fea20) is not the object's thread (0x1fa15850).
Cannot move to target thread (0x1f9fea20)

QObject::moveToThread: Current thread (0x1f9fea20) is not the object's thread (0x1fa15850).
Cannot move to target thread (0x1f9fea20)

QObject::moveToThread: Current thread (0x1f9fea20) is not the object's thread (0x1fa15850).
Cannot move to target thread (0x1f9fea20)

QObject::moveToThread: Current thread (0x1f9fea20) is not the object's thread (0x1fa15850).
Cannot move to target thread (0x1f9fea20)

QObject::moveToThread: Current thread (0x1f9fea20) is not the object's thread (0x1fa15850).
Cannot move to target thread (0x1f9fea20)

QObject::moveToThread: Current thread (0x1f9fea20) is not the object's thread (0x1fa15850).
Cannot move to target thread (0x1f9fea20)

QObject::moveToThread: Current thread (0x1f9fea20) is n